# 03 Evaluación y Fine-tuning — A.U.R.A

Este notebook evalúa el **modelo supervisado (ML tabular)** y realiza fine-tuning ligero.

> NLP y reglas están en el notebook 02 y en `src/nlp` + `src/rules`. El ML **no** incluye `max_similitud_narrativa` (PDF §9: capas separadas).

## Qué entrega
- Métricas: **AUC-ROC**, **Precision/Recall/F1**, matriz de confusión.
- Curvas: ROC y Precision-Recall.
- Selección de **threshold** según objetivo del reto (priorizar revisión humana).
- Fine-tuning simple de LogisticRegression (C) con validación cruzada.
- Explicabilidad: top features por coeficiente.


In [10]:
from pathlib import Path
import sys

# Permite importar `src.*` aunque ejecutes desde /notebooks
ROOT = Path("..").resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd

from src.dataio.csv_exports import get_latest_export_dir, load_exported_tables

from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    roc_auc_score,
    classification_report,
    confusion_matrix,
    precision_recall_curve,
    roc_curve,
    average_precision_score,
)
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression

import plotly.express as px
import plotly.graph_objects as go

EXPORT_DIR = get_latest_export_dir(Path("..") / "data" / "raw" / "supabase_export")
print(f"✅ Usando export CSV: {EXPORT_DIR}")


✅ Usando export CSV: ..\data\raw\supabase_export\20260528_094805


In [11]:
tables = load_exported_tables(EXPORT_DIR)

df_siniestros = tables["siniestros"]
df_polizas = tables["polizas"]
df_proveedores = tables["proveedores"]
df_documentos = tables["documentos"]

for c in ['fecha_ocurrencia','fecha_reporte']:
    if c in df_siniestros.columns:
        df_siniestros[c] = pd.to_datetime(df_siniestros[c], errors='coerce')

print('Siniestros:', len(df_siniestros))


Siniestros: 1000


In [12]:
# Build df master (mismo set de features que Notebook 02)

df = df_siniestros.merge(
    df_polizas[['id_poliza','monto_asegurado','tipo_cobertura','ramo','ciudad','estado_poliza','deducible','prima']]
        if 'id_poliza' in df_polizas.columns else df_polizas,
    on='id_poliza',
    how='left',
    suffixes=('', '_pol')
)

df = df.merge(
    df_proveedores[['id_proveedor','tipo_proveedor','ciudad','lista_restrictiva','proveedor_preferente']]
        if 'id_proveedor' in df_proveedores.columns else df_proveedores,
    on='id_proveedor',
    how='left',
    suffixes=('', '_prov')
)

# documentos -> flags agregados
if not df_documentos.empty and 'id_siniestro' in df_documentos.columns:
    doc = df_documentos.copy()
    for c in ['posible_adulteracion','inconsistencia_detectada','es_legible','entregado']:
        if c not in doc.columns:
            doc[c] = False

    agg = doc.groupby('id_siniestro').agg(
        doc_count=('id_documento', 'count'),
        doc_inconsistencia=('inconsistencia_detectada', 'max'),
        doc_adulteracion=('posible_adulteracion', 'max'),
        doc_entregado=('entregado', 'min'),
        doc_es_legible=('es_legible', 'min'),
    )
    agg['doc_ilegible'] = (~agg['doc_es_legible'].astype(bool)).astype(int)
    agg['doc_faltante'] = (~agg['doc_entregado'].astype(bool)).astype(int)

    df = df.merge(agg.reset_index(), on='id_siniestro', how='left')

# Features
if {'monto_reclamado','monto_asegurado'}.issubset(df.columns):
    df['ratio_reclamado'] = df['monto_reclamado'] / df['monto_asegurado']

df['delta_notificacion_dias'] = df.get('dias_entre_ocurrencia_reporte', np.nan)
df['delta_vigencia_dias'] = df.get('dias_desde_inicio_poliza', np.nan)

for c in ['doc_count','doc_inconsistencia','doc_adulteracion','doc_ilegible','doc_faltante']:
    if c in df.columns:
        df[c] = df[c].fillna(0)

print('✅ df listo (features tabulares para ML):', df.shape)


✅ df listo (features tabulares para ML): (1000, 53)


In [13]:
# Dataset ML
label = 'etiqueta_fraude_simulada'

feature_cols_num = [
    'monto_reclamado','monto_estimado','ratio_reclamado',
    'delta_notificacion_dias','delta_vigencia_dias',
    'historial_siniestros_asegurado',
    'doc_count','doc_inconsistencia','doc_adulteracion','doc_ilegible','doc_faltante',
]
feature_cols_cat = ['cobertura','sucursal','tipo_proveedor']
feature_cols_bool = ['documentos_completos','lista_restrictiva','proveedor_preferente']

for c in feature_cols_num:
    if c not in df.columns:
        df[c] = np.nan
for c in feature_cols_cat:
    if c not in df.columns:
        df[c] = None
for c in feature_cols_bool:
    if c not in df.columns:
        df[c] = False

X = df[feature_cols_num + feature_cols_cat + feature_cols_bool].copy()
for c in feature_cols_bool:
    X[c] = X[c].fillna(False).astype(int)

y = df[label].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pre = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[
            ('imp', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
        ]), feature_cols_num),
        ('cat', Pipeline(steps=[('imp', SimpleImputer(strategy='most_frequent')),
                               ('oh', OneHotEncoder(handle_unknown='ignore'))]), feature_cols_cat),
        ('bool', Pipeline(steps=[('imp', SimpleImputer(strategy='most_frequent'))]), feature_cols_bool),
    ]
)

print('Train size:', len(X_train), 'Test size:', len(X_test), 'Fraude rate train:', y_train.mean())


Train size: 750 Test size: 250 Fraude rate train: 0.13466666666666666


In [14]:
# 1) Evaluación base (LogisticRegression)

base_model = Pipeline(steps=[
    ('pre', pre),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', C=1.0))
])

base_model.fit(X_train, y_train)
proba = base_model.predict_proba(X_test)[:, 1]

auc = roc_auc_score(y_test, proba)
ap = average_precision_score(y_test, proba)
print('AUC-ROC:', auc)
print('Average Precision (PR AUC):', ap)

# Threshold default
thr = 0.5
pred = (proba >= thr).astype(int)
print('\nReporte (threshold=0.5)')
print(classification_report(y_test, pred, zero_division=0))
print('Matriz confusión:\n', confusion_matrix(y_test, pred))


AUC-ROC: 0.9305555555555556
Average Precision (PR AUC): 0.805978962068967

Reporte (threshold=0.5)
              precision    recall  f1-score   support

           0       0.97      0.88      0.92       216
           1       0.53      0.85      0.65        34

    accuracy                           0.88       250
   macro avg       0.75      0.87      0.79       250
weighted avg       0.91      0.88      0.89       250

Matriz confusión:
 [[190  26]
 [  5  29]]


In [15]:
# 2) Curvas ROC y Precision-Recall

fpr, tpr, thr_roc = roc_curve(y_test, proba)
prec, rec, thr_pr = precision_recall_curve(y_test, proba)

fig = go.Figure()
fig.add_trace(go.Scatter(x=fpr, y=tpr, mode='lines', name='ROC'))
fig.add_trace(go.Scatter(x=[0,1], y=[0,1], mode='lines', name='baseline', line=dict(dash='dash')))
fig.update_layout(title='ROC curve', xaxis_title='FPR', yaxis_title='TPR')
fig.show()

fig = go.Figure()
fig.add_trace(go.Scatter(x=rec, y=prec, mode='lines', name='PR'))
fig.update_layout(title='Precision-Recall curve', xaxis_title='Recall', yaxis_title='Precision')
fig.show()


In [16]:
# 3) Selección de threshold según objetivo
# Objetivo hackathon: alta cobertura (recall) para revisión humana.
# Nota: `proba` es del modelo base (C=1.0). Tras CV (celda siguiente) el modelo prod usa C=0.1.
# Umbral operativo fijo en prod (`scripts/train_fraud_model.py`): 0.7

thresholds = np.linspace(0.05, 0.95, 19)
rows = []
for t in thresholds:
    p = (proba >= t).astype(int)
    cm = confusion_matrix(y_test, p)
    tn, fp, fn, tp = cm.ravel()
    precision = tp / (tp + fp + 1e-9)
    recall = tp / (tp + fn + 1e-9)
    f1 = 2 * precision * recall / (precision + recall + 1e-9)
    rows.append({'thr': t, 'precision_fraude': precision, 'recall_fraude': recall, 'f1_fraude': f1, 'fp': fp, 'fn': fn})

thr_df = pd.DataFrame(rows)
display(thr_df.sort_values('f1_fraude', ascending=False).head(10))

# Ejemplo: threshold que maximiza F1
best_thr = float(thr_df.sort_values('f1_fraude', ascending=False).iloc[0]['thr'])
print('✅ Threshold recomendado (max F1):', best_thr)


,thr,precision_fraude,recall_fraude,f1_fraude,fp,fn
13,0.70,0.750000,0.705882,0.727273,8,10
14,0.75,0.814815,0.647059,0.721311,5,12
15,0.80,0.869565,0.588235,0.701754,3,14
16,0.85,0.947368,0.529412,0.679245,1,16
12,0.65,0.648649,0.705882,0.676056,13,10
10,0.55,0.574468,0.794118,0.666667,20,7
11,0.60,0.577778,0.764706,0.658228,19,8
9,0.50,0.527273,0.852941,0.651685,26,5
17,0.90,0.941176,0.470588,0.627451,1,18
8,0.45,0.461538,0.882353,0.606061,35,4


✅ Threshold recomendado (max F1): 0.7


In [17]:
# 4) Fine-tuning ligero: C de LogisticRegression con CV (AUC)

Cs = [0.1, 0.3, 1.0, 3.0, 10.0]
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

rows = []
for C in Cs:
    aucs = []
    for train_idx, val_idx in skf.split(X_train, y_train):
        Xt, Xv = X_train.iloc[train_idx], X_train.iloc[val_idx]
        yt, yv = y_train.iloc[train_idx], y_train.iloc[val_idx]

        m = Pipeline(steps=[
            ('pre', pre),
            ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', C=C))
        ])
        m.fit(Xt, yt)
        pv = m.predict_proba(Xv)[:,1]
        aucs.append(roc_auc_score(yv, pv))

    rows.append({'C': C, 'auc_cv_mean': float(np.mean(aucs)), 'auc_cv_std': float(np.std(aucs))})

cv = pd.DataFrame(rows).sort_values('auc_cv_mean', ascending=False)
display(cv)

best_C = float(cv.iloc[0]['C'])
print('✅ Mejor C por CV AUC:', best_C)

# Reentrenar con best_C y evaluar
final_model = Pipeline(steps=[
    ('pre', pre),
    ('clf', LogisticRegression(max_iter=2000, class_weight='balanced', C=best_C))
])
final_model.fit(X_train, y_train)
final_proba = final_model.predict_proba(X_test)[:,1]

print('AUC final:', roc_auc_score(y_test, final_proba))
print('PR AUC final:', average_precision_score(y_test, final_proba))

# Alineación con prod: mismo C=0.1; umbral operativo fijo 0.7 (no max F1 del holdout C=0.1)
OPERATIONAL_THR = 0.7
pred_op = (final_proba >= OPERATIONAL_THR).astype(int)
print(f'\nReporte modelo final (C={best_C}, threshold operativo={OPERATIONAL_THR}):')
print(classification_report(y_test, pred_op, zero_division=0))
print('(Prod guarda este threshold en fraud_model_config.json)')


,C,auc_cv_mean,auc_cv_std
0,0.1,0.892116,0.067586
1,0.3,0.886227,0.070976
2,1.0,0.882135,0.073026
3,3.0,0.879013,0.073259
4,10.0,0.876745,0.073463


✅ Mejor C por CV AUC: 0.1
AUC final: 0.9328703703703705
PR AUC final: 0.811188837102538


In [18]:
# 5) Explicabilidad: top features (coeficientes)

# Obtener nombres de features tras OneHot
pre_fitted = final_model.named_steps['pre']

feature_names = []
# num
feature_names += feature_cols_num
# cat
oh = pre_fitted.named_transformers_['cat'].named_steps['oh']
cat_names = list(oh.get_feature_names_out(feature_cols_cat))
feature_names += cat_names
# bool
feature_names += feature_cols_bool

coef = final_model.named_steps['clf'].coef_.ravel()
coef_df = pd.DataFrame({'feature': feature_names, 'coef': coef}).sort_values('coef', ascending=False)

print('Top 15 features que suben riesgo (coef +):')
display(coef_df.head(15))
print('Top 15 features que bajan riesgo (coef -):')
display(coef_df.tail(15))


Top 15 features que suben riesgo (coef +):


,feature,coef
5,historial_siniestros_asegurado,1.450528
12,cobertura_DAÑOS,0.398580
0,monto_reclamado,0.367386
3,delta_notificacion_dias,0.353069
14,sucursal_CUENCA,0.192832
19,tipo_proveedor_PERITO AUTOMOTRIZ,0.136182
1,monto_estimado,0.122584
17,sucursal_QUITO,0.103800
6,doc_count,0.093313
2,ratio_reclamado,0.021005


Top 15 features que bajan riesgo (coef -):


,feature,coef
2,ratio_reclamado,0.021005
18,tipo_proveedor_CENTRO DE COLISIÓN,0.003705
7,doc_inconsistencia,0.000000
10,doc_faltante,0.000000
23,proveedor_preferente,0.000000
22,lista_restrictiva,-0.058352
15,sucursal_GUAYAQUIL,-0.083076
11,cobertura_CHOQUE,-0.119392
20,tipo_proveedor_TALLER MECÁNICO,-0.141979
8,doc_adulteracion,-0.179811
